In [16]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

In [17]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_text(prompt, max_length=4000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

Loading checkpoint shards: 100%|██████████| 3/3 [00:20<00:00,  6.89s/it]
Some parameters are on the meta device because they were offloaded to the disk and cpu.


In [18]:
pdf_path = "./dataset/cv1.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)
vectordb = FAISS.from_documents(chunks, embedding)

In [19]:
full_name_schema = ResponseSchema(
    name="full_name",
    description="The full legal name of the applicant."
)
email_schema = ResponseSchema(
    name="email",
    description="The email address of the applicant."
)
education_schema = ResponseSchema(
    name="education",
    description="A list of the education that the applicant completed with degree, institution and year of graduation."
)
skills_schema = ResponseSchema(
    name="skills",
    description="A list of all skills stated in applicant resume."
)
experience_schema = ResponseSchema(
    name="experience",
    description="A list of the experiences that the applicant had including role, compony and the period of employment."
)

response_schemas = [full_name_schema,
                    email_schema,
                    education_schema,
                    skills_schema, 
                    experience_schema]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

resume_extraction_template = """
You are a senior human resouses manager that extracts qualifications out of applicants' resumes.

Extract all qualifications as following:
full name
email address
education containing degree, the institution from which the dgree got issued and the year of graduation 
skills containing all major skills the applicant have
experience containing role name, company name and the period of employment.


Respond ONLY in JSON format as follows:
{format_instructions}

Example Input:
"
John Smith – john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020–2023)
- Data Scientist at OpenAI (2023–Present)
"

Now extract from the following input:
"{user_input}"
"""

In [20]:
def ask_question(query):
    docs = vectordb.similarity_search(query, k=3)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    prompt = f"""You are a helpful assistant. Use the following context to answer the question. Context: {context} Question: {query} Answer:"""
    
    result = generate_text(prompt)
    return result.strip()

In [21]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"

In [ ]:
user_input = "Analyze this resume."

prompt = PromptTemplate(
    template=resume_extraction_template,
    input_variables=["user_input", "format_instructions"]
).format(user_input=user_input, format_instructions=format_instructions)

: 

In [ ]:
answer = ask_question(prompt)
print("\n Answer:", answer.split("Answer:")[-1], "\n")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
json_text = extract_json_block(answer)
print(json_text)

```json
{
	"full_name": "Mohamed Matar",
	"email": "+20 01019729232|Email",
	"education": [
		{
			"degree": "Bachelor Degree",
			"institution": "Misr International University",
			"year": "Jul. 2021 – Jul. 2025"
		}
	],
	"skills": "Python, C#, TensorFlow, Keras, Scikit-learn, Pandas, NumPy, Matplotlib, Git, Github, Jupyter, Google Colab, VS Code, Visual Studio",
	"experience": [
		{
			"role": "Machine Learning Intern",
			"company": "Prodigy InfoTech Mumbai, India",
			"period": "May 2024 – June 2024"
		},
		{
			"role": "Projects",
			"company": "Multiple",
			"period": ["September 2024 – June 2025", "July 2025 – October 2025", "May 2024 – July 2024"]
		}
	]
}
```


In [ ]:
output_data = output_parser.parse(json_text)

In [ ]:
print(output_data)

{'full_name': 'Mohamed Matar', 'email': '+20 01019729232|Email', 'education': [{'degree': 'Bachelor Degree', 'institution': 'Misr International University', 'year': 'Jul. 2021 – Jul. 2025'}], 'skills': 'Python, C#, TensorFlow, Keras, Scikit-learn, Pandas, NumPy, Matplotlib, Git, Github, Jupyter, Google Colab, VS Code, Visual Studio', 'experience': [{'role': 'Machine Learning Intern', 'company': 'Prodigy InfoTech Mumbai, India', 'period': 'May 2024 – June 2024'}, {'role': 'Projects', 'company': 'Multiple', 'period': ['September 2024 – June 2025', 'July 2025 – October 2025', 'May 2024 – July 2024']}]}


In [ ]:
print(output_data["education"])

[{'degree': 'Bachelor Degree', 'institution': 'Misr International University', 'year': 'Jul. 2021 – Jul. 2025'}]


In [ ]:
model.save_pretrained("/kaggle/working/Mistral-7B-Instruct-v0.2")
tokenizer.save_pretrained("/kaggle/working/Mistral-7B-Instruct-v0.2")

('/kaggle/working/Mistral-7B-Instruct-v0.2/tokenizer_config.json',
 '/kaggle/working/Mistral-7B-Instruct-v0.2/special_tokens_map.json',
 '/kaggle/working/Mistral-7B-Instruct-v0.2/chat_template.jinja',
 '/kaggle/working/Mistral-7B-Instruct-v0.2/tokenizer.model',
 '/kaggle/working/Mistral-7B-Instruct-v0.2/added_tokens.json',
 '/kaggle/working/Mistral-7B-Instruct-v0.2/tokenizer.json')

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/Mistral-7B-Instruct-v0.2",
    "zip",
    "/kaggle/working/Mistral-7B-Instruct-v0.2"
)